In [90]:
import pandas as pd
import numpy as np
import re

In [81]:
viaticos = pd.read_csv('registraduria.csv', dtype=str)
viaticos.columns = viaticos.columns.str.strip()

print(f"Filas    : {len(viaticos):,}")
print(f"Columnas : {list(viaticos.columns)}")

Filas    : 16,001
Columnas : ['commission_request_id', 'procedure_type', 'employee_id', 'employee_name', 'employee_position', 'commission_days', 'origin', 'destination', 'start_date', 'end_date', 'total_travel_allowance', 'commission_purpose', 'year', 'origen_register']


In [82]:
df.head(3)

,commission_request_id,procedure_type,employee_id,employee_name,employee_position,commission_days,origin,destination,start_date,end_date,total_travel_allowance,commission_purpose,year,origen_register
0,2201097,Inicial,16699992,JAIME ESCOBAR VELEZ,Gerente Grado (4),1.5,BOGOTA D.C.,MANIZALES,2022-01-09,2022-01-10,1610649.0,Realizar visita a la obra de la nueva sede de ...,2022,Ficticio generado a partir del patrón 2025
1,2201941,Inicial,16699992,JAIME ESCOBAR VELEZ,Gerente Grado (4),0.5,BOGOTA D.C.,MEDELLIN,2022-01-21,2022-01-21,443812.0,Visitar inmuebles para la posible adquisicion ...,2022,Ficticio generado a partir del patrón 2025
2,2203180,Inicial,1061708756,ANGELA MARIA NAVARRO PERALTA,Director Grado (3),1.5,BOGOTA D.C.,CUCUTA,2022-01-21,2022-01-22,1343765.0,"Visita de verificación, recibo físico y de bie...",2022,Ficticio generado a partir del patrón 2025


In [83]:
viaticos["commission_request_id"]  = viaticos["commission_request_id"].str.replace(r"\.0$", "", regex=True)
viaticos["total_travel_allowance"] = pd.to_numeric(viaticos["total_travel_allowance"], errors="coerce")
viaticos["commission_days"]        = pd.to_numeric(viaticos["commission_days"],        errors="coerce")
viaticos["start_date"] = pd.to_datetime(viaticos["start_date"], dayfirst=True, errors="coerce")
viaticos["end_date"]   = pd.to_datetime(viaticos["end_date"],   dayfirst=True, errors="coerce")

print(viaticos.dtypes)
print(f"\nNulos:\n{viaticos.isnull().sum()}")

commission_request_id             object
procedure_type                    object
employee_id                       object
employee_name                     object
employee_position                 object
commission_days                  float64
origin                            object
destination                       object
start_date                datetime64[ns]
end_date                  datetime64[ns]
total_travel_allowance           float64
commission_purpose                object
year                              object
origen_register                   object
dtype: object

Nulos:
commission_request_id     0
procedure_type            0
employee_id               0
employee_name             0
employee_position         0
commission_days           0
origin                    0
destination               0
start_date                0
end_date                  0
total_travel_allowance    0
commission_purpose        0
year                      0
origen_register           0
dtype: int64

In [84]:
# Mapa cargo → dependencia
def inferir_dependencia(cargo: str) -> str:
    cargo = str(cargo).lower()
    if "contralor delegado" in cargo or "contralora delegada" in cargo:
        return "Contraloría Delegada"
    if "gerente" in cargo:
        return "Gerencia Departamental"
    if "director" in cargo:
        return "Dirección de Vigilancia Fiscal"
    if "coordinador" in cargo or "coordinadora" in cargo:
        return "Coordinación de Gestión"
    if "profesional especializado" in cargo:
        return "Oficina de Planeación"
    if "profesional universitario" in cargo:
        return "Dirección de Gestión del Talento Humano"
    if "auxiliar" in cargo:
        return "Secretaría General"
    if "asesor" in cargo:
        return "Despacho del Contralor"
    if "contratista" in cargo:
        return "Oficina de Contratación"
    if "provincial" in cargo:
        return "Contraloría Provincial"
    return "Secretaría General"

funcionarios = (
    df[["employee_id", "employee_name", "employee_position"]]
    .drop_duplicates(subset="employee_id", keep="last")
    .reset_index(drop=True)
)

funcionarios["dependency"] = funcionarios["employee_position"].apply(inferir_dependencia)

funcionarios.head(5)

,employee_id,employee_name,employee_position,dependency
0,81458772,KEVIN YULIANA RINCON RINCON,Profesional Universitario Grado (2),Dirección de Gestión del Talento Humano
1,42779491,VIVIANA FELIPE CASTANEDA RUIZ,Profesional Universitario Grado (1),Dirección de Gestión del Talento Humano
2,14242158,LEONARDO JORGE RAMIREZ SILVA,Profesional Especializado Grado (3),Oficina de Planeación
3,43522920,MATEO PAULA ACOSTA RINCON,Profesional Universitario Grado (2),Dirección de Gestión del Talento Humano
4,31721032,MARCELA RICARDO NAVARRO SUAREZ,Profesional Especializado Grado (3),Oficina de Planeación


In [85]:
municipios = pd.read_csv('divipola.csv', dtype=str)

municipios = municipios.rename(columns={
    "Código Departamento": "department_code",
    "Nombre Departamento": "department_name",
    "Código Municipio": "municipality_code",
    "Nombre Municipio": "municipality_name",
    "Tipo: Municipio / Isla / Área no municipalizada": "admin_type",
    "longitud": "longitude",
    "Latitud": "latitude",
})

In [86]:
municipios["municipality_code"] = pd.to_numeric(municipios["municipality_code"], errors="coerce")
municipios["department_code"] = pd.to_numeric(municipios["department_code"], errors="coerce")
municipios['longitude'] = municipios['longitude'].astype(str).str.replace(',', '.')
municipios['latitude'] = municipios['latitude'].astype(str).str.replace(',', '.')
municipios["longitude"] = pd.to_numeric(municipios["longitude"], errors="coerce")
municipios["latitude"] = pd.to_numeric(municipios["latitude"], errors="coerce")

print(municipios.dtypes)
print(f"\nNulos:\n{municipios   .isnull().sum()}")

department_code        int64
department_name       object
municipality_code      int64
municipality_name     object
admin_type            object
longitude            float64
latitude             float64
dtype: object

Nulos:
department_code      0
department_name      0
municipality_code    0
municipality_name    0
admin_type           0
longitude            0
latitude             0
dtype: int64


In [87]:
funcionarios.to_csv( "funcionarios.csv", encoding='latin1', index=False)

In [88]:
rubros = [
    "Gastos de Viaje y Transporte - Nivel Central",
    "Gastos de Viaje y Transporte - Gerencias Departamentales",
    "Comisiones al Interior del País - Funcionarios",
    "Comisiones al Interior del País - Contratistas",
    "Viáticos y Gastos de Viaje - Auditorías",
    "Viáticos Control Fiscal Participativo",
    "Viáticos Estrategia Compromiso Colombia",
    "Gastos de Desplazamiento - Visitas Técnicas",
    "Viáticos Indagaciones Preliminares",
    "Gastos de Transporte Aéreo",
    "Gastos de Transporte Terrestre",
    "Viáticos Interventoría de Contratos",
    "Comisiones Especiales - Despacho Contralor",
    "Viáticos Proceso de Responsabilidad Fiscal",
    "Gastos de Representación y Protocolo",
]

total_viaticos = df["total_travel_allowance"].sum()          # Total real del CSV
apropiado_base = total_viaticos / len(rubros)                 # Distribuimos el total

presupuesto_rows = []
for i, rubro in enumerate(rubros):
    pid          = f"PPTO{i+1:04d}"
    factor = random.uniform(1.15, 1.40)  # apropiado > ejecutado
    apropiado    = round(apropiado_base * factor, -3)          # redondeo a miles
    saldo        = round(apropiado * random.uniform(0.05, 0.30), -3)
    presupuesto_rows.append({
        "id_presupuesto"  : pid,
        "rubro"           : rubro,
        "valor_apropiado" : apropiado,
        "saldo_disponible": saldo,
    })

presupuesto = pd.DataFrame(presupuesto_rows)
presupuesto

,id_presupuesto,rubro,valor_apropiado,saldo_disponible
0,PPTO0001,Gastos de Viaje y Transporte - Nivel Central,1.522054e+09,124018000.0
1,PPTO0002,Gastos de Viaje y Transporte - Gerencias Depar...,1.389749e+09,340340000.0
2,PPTO0003,Comisiones al Interior del País - Funcionarios,1.544034e+09,78897000.0
3,PPTO0004,Comisiones al Interior del País - Contratistas,1.604008e+09,98850000.0
4,PPTO0005,Viáticos y Gastos de Viaje - Auditorías,1.572566e+09,188025000.0
5,PPTO0006,Viáticos Control Fiscal Participativo,1.545544e+09,155316000.0
6,PPTO0007,Viáticos Estrategia Compromiso Colombia,1.346447e+09,228827000.0
7,PPTO0008,Gastos de Desplazamiento - Visitas Técnicas,1.453410e+09,194882000.0
8,PPTO0009,Viáticos Indagaciones Preliminares,1.598214e+09,168568000.0
9,PPTO0010,Gastos de Transporte Aéreo,1.544280e+09,352202000.0


In [89]:
viaticos["commission_id"] = pd.NA
viaticos = viaticos.drop(columns=["employee_name", "employee_position"], errors="ignore")
viaticos.head()

,commission_request_id,procedure_type,employee_id,commission_days,origin,destination,start_date,end_date,total_travel_allowance,commission_purpose,year,origen_register,commission_id
0,2201097,Inicial,16699992,1.5,BOGOTA D.C.,MANIZALES,2022-01-09,2022-01-10,1610649.0,Realizar visita a la obra de la nueva sede de ...,2022,Ficticio generado a partir del patrón 2025,<NA>
1,2201941,Inicial,16699992,0.5,BOGOTA D.C.,MEDELLIN,2022-01-21,2022-01-21,443812.0,Visitar inmuebles para la posible adquisicion ...,2022,Ficticio generado a partir del patrón 2025,<NA>
2,2203180,Inicial,1061708756,1.5,BOGOTA D.C.,CUCUTA,2022-01-21,2022-01-22,1343765.0,"Visita de verificación, recibo físico y de bie...",2022,Ficticio generado a partir del patrón 2025,<NA>
3,2203264,Inicial,1136883466,1.5,BOGOTA D.C.,CUCUTA,2022-01-21,2022-01-22,549239.0,"Visita de verificación técnica, recibo físico ...",2022,Ficticio generado a partir del patrón 2025,<NA>
4,2200958,Inicial,1088253721,1.5,BOGOTA D.C.,CALI,2022-01-22,2022-01-23,1378698.0,Asistir en representacion del Contralor Genera...,2022,Ficticio generado a partir del patrón 2025,<NA>


In [91]:
def classify_commission(purpose):
    if pd.isna(purpose):
        return None
    purpose = str(purpose).lower()
    
    if 'indagación preliminar' in purpose or 'indagacion preliminar' in purpose or ' ip ' in purpose:
        return 'PPTO0009'
    if 'responsabilidad fiscal' in purpose or ' prf ' in purpose:
        return 'PPTO0014'
    if 'interventoría' in purpose or 'interventoria' in purpose:
        return 'PPTO0012'
    if 'compromiso colombia' in purpose:
        return 'PPTO0007'
    if 'participativo' in purpose or 'consulta previa' in purpose:
        return 'PPTO0006'
    if 'contralor general' in purpose or 'despacho' in purpose:
        return 'PPTO0013'
    if 'auditoría' in purpose or 'auditoria' in purpose or 'visita fiscal' in purpose:
        return 'PPTO0005'
    if 'visita técnica' in purpose or 'visita tecnica' in purpose or 'verificación técnica' in purpose or 'verificacion tecnica' in purpose or 'visita de verificación' in purpose or 'visita de verificacion' in purpose:
        return 'PPTO0008'
    if 'protocolo' in purpose or 'representación' in purpose or 'representacion' in purpose:
        return 'PPTO0015'
    if 'transporte terrestre' in purpose or 'vehículo' in purpose or 'vehiculo' in purpose or 'conductor' in purpose:
        return 'PPTO0011'
    if 'transporte aéreo' in purpose or 'transporte aereo' in purpose or 'aéreo' in purpose or 'aereo' in purpose:
        return 'PPTO0010'
    if 'gerencia departamental' in purpose or 'gerencias departamentales' in purpose:
        return 'PPTO0002'
    if 'nivel central' in purpose or 'sede central' in purpose:
        return 'PPTO0001'

    return 'PPTO0003'


viaticos['commission_id'] = viaticos['commission_purpose'].apply(classify_commission)
viaticos.to_csv('Commission_ID_Populated.csv', index=False)

print(viaticos[['commission_purpose', 'commission_id']].head(10))
print("\nValue counts for commission_id:")
print(viaticos['commission_id'].value_counts())

                                  commission_purpose commission_id
0  Realizar visita a la obra de la nueva sede de ...      PPTO0002
1  Visitar inmuebles para la posible adquisicion ...      PPTO0002
2  Visita de verificación, recibo físico y de bie...      PPTO0008
3  Visita de verificación técnica, recibo físico ...      PPTO0008
4  Asistir en representacion del Contralor Genera...      PPTO0013
5  Realizar acompañamiento prestando los servicio...      PPTO0011
6  Asistir a la invitación a la celebracion del d...      PPTO0011
7  Asistir a la invitación a la celebracion del d...      PPTO0011
8  Asistir a reunión de seguimiento proyecto Cana...      PPTO0008
9  Ejecutar visita técnica a las actividades del ...      PPTO0009

Value counts for commission_id:
commission_id
PPTO0003    8885
PPTO0014    1560
PPTO0008    1388
PPTO0005    1193
PPTO0009    1110
PPTO0002     421
PPTO0006     396
PPTO0011     342
PPTO0010     208
PPTO0013     180
PPTO0007     151
PPTO0012      92
PPTO0015    